<!-- dads-lab-header -->
<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M05/M05_Lab1_Embeddings_VectorStores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🧬 M05 Lab 1 — Embeddings & Vector Stores</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~40 min</p>
</div>

> **📌 Note on models.** This lab references specific LLM versions through `DEFAULT_CHAT_MODEL` and `DEFAULT_MINI_MODEL` constants in the `dads5250` utility package. Models update quickly — feel free to swap to any newer OpenAI / Anthropic / Google model you have access to.


In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M05 Lab 1 — Embeddings & Vector Stores')            # sticky banner so you always see which lab you're in
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works


![RAG Intro Lab](https://www.dropbox.com/scl/fi/hyyhk4rkslkgi6rd1ki5z/RAG_Intro_Lab.png?rlkey=31iuzmmvt9ta3xf649jma4y67&raw=1)


In [ ]:
# 📦 Installing Required Libraries for LangChain RAG Lab (Quiet Mode)
# ==================================================

# --- Core LangChain and OpenAI Integration ---
!pip install -q --upgrade langchain langchain-community langchain-openai langchain_text_splitters langchain-core

# --- OpenAI SDK ---
# 'openai': Required to access GPT-3.5/4 and manage API keys, works with both LangChain and direct calls
!pip install -q --upgrade openai

# --- Vector Databases for Retrieval (RAG) ---
# 'faiss-cpu': Facebook's FAISS for fast vector search (in-memory or persistent)
# 'chromadb': Lightweight vector database, ideal for local demos and quick setup
!pip install -q --upgrade faiss-cpu chromadb

# --- Tokenization and Unstructured Data Support ---
# 'tiktoken': Fast, efficient tokenizer (used with OpenAI, supports counting tokens accurately)
# 'unstructured': Loads/cleans data from PDFs, DOCX, HTML, email, etc. for use in retrieval pipelines
# 'unstructured[pdf]': Adds PDF parsing support (using pdfminer, pypdf, etc.)
# 'pypdf', 'pdfminer.six': Popular PDF parsing backends, required for some document loaders
!pip install -q --upgrade tiktoken unstructured "unstructured[pdf]" pypdf pdfminer.six


In [ ]:
# 📚 LangChain RAG Lab: Library Imports & Setup
# ====================================================
# ✅ This cell handles all required imports, grouped by category for clarity.

# 🧱 System & Environment Setup
import os  # Environment variable access
import requests  # For fetching remote resources (e.g., PDFs, data files)
from google.colab import userdata  # Accessing Colab-specific secure data

# 🧪 Jupyter & Colab Display Utilities
import ipywidgets as widgets  # Interactive widgets
from IPython.display import clear_output, display, HTML  # Display controls

# 🔑 OpenAI API
import openai  # Optional: raw API access (not required for LangChain unless custom use)

# 🧠 LangChain Core Modules
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LLM + Embeddings via OpenAI
from langchain_core.prompts import PromptTemplate  # Structured prompt templates
# from langchain_classic.memory import ConversationBufferMemory  # For chat history memory


# ✅ Confirmation
print("✅ All libraries imported and categorized successfully!")


<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.section-header {
    background-color: #e8f0fe;
    padding: 12px;
    margin: 12px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

.description-box {
    background-color: #f8f9fa;
    padding: 14px;
    margin: 12px 0;
    border-radius: 4px;
    border: 1px solid #e0e0e0;
}

h2 {
    color: #1a73e8;
    font-size: 1.15em;
    margin: 0;
}

.code-note {
    background-color: #fff;
    padding: 10px;
    margin: 10px 0;
    border-radius: 4px;
    border: 1px solid #dadce0;
    font-size: 0.9em;
}

.highlight {
    color: #1a73e8;
    font-weight: 600;
}
</style>
</head>
<body>

<div class="section-header">
    <h2>🖨️ Pretty Print Function</h2>
</div>

<div class="description-box">
    <p style="margin: 0 0 8px 0;">The <span class="highlight">pretty_print()</span> function enhances output readability by transforming standard text into styled HTML blocks. This utility function replaces basic print statements with visually appealing formatted displays that improve the user experience when viewing model responses and system outputs.</p>
    
    <p style="margin: 8px 0 0 0;">Key features include automatic detection and formatting of bulleted lists, proper line break handling, and consistent visual styling that matches the laboratory's design theme. The function accepts two parameters: the text content to display and an optional title that appears as a header above the formatted output.</p>
</div>

<div class="code-note">
    <strong>Usage:</strong> Replace standard <code>print()</code> statements with <code>pretty_print()</code> throughout your notebook to maintain consistent, professional output formatting.
</div>

</body>
</html>

<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.section-header {
    background-color: #e8f0fe;
    padding: 12px;
    margin: 12px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

h2 {
    color: #1a73e8;
    font-size: 1.15em;
    margin: 0;
}
</style>
</head>
<body>

<div class="section-header">
    <h2>🔑 OpenAI API Key Setup from Colab Secrets</h2>
</div>

</body>
</html>

<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.section-header {
    background-color: #e8f0fe;
    padding: 12px;
    margin: 12px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

h2 {
    color: #1a73e8;
    font-size: 1.15em;
    margin: 0;
}
</style>
</head>
<body>

<div class="section-header">
    <h2>🔷 Part 1: Non-RAG Model Implementation</h2>
</div>

</body>
</html>

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# SIMPLE LANGCHAIN LLM QUERY - NO RAG COMPONENTS
# Direct language model query using LangChain without document retrieval
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Initialize language model
llm = ChatOpenAI(
    model=DEFAULT_MINI_MODEL,
    temperature=0.7,
    max_tokens=500
)

# Execute query
query = "What do I learn in the GenAI course in 3 bullets, software, application. Also who is the prof? any hints for me to gain a good grade?"
response = llm.invoke(query)

# Display result
pretty_print(response.content, title="🎯 Direct LLM Response")

<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.section-header {
    background-color: #e8f0fe;
    padding: 12px;
    margin: 12px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

.explanation-box {
    background-color: #f0f6ff;
    border-left: 3px solid #1a73e8;
    padding: 16px;
    margin: 16px 0;
    border-radius: 4px;
}

.process-diagram {
    background: white;
    border: 1px solid #dadce0;
    border-radius: 8px;
    padding: 20px;
    margin: 16px 0;
}

.step-container {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
    margin: 20px 0;
}

.step-card {
    background: #f8f9fa;
    border-radius: 8px;
    padding: 15px;
    text-align: center;
    border-top: 3px solid #1a73e8;
    position: relative;
}

.step-number {
    background: #1a73e8;
    color: white;
    width: 28px;
    height: 28px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: bold;
    margin: 0 auto 10px;
}

.step-arrow {
    position: absolute;
    right: -10px;
    top: 50%;
    transform: translateY(-50%);
    color: #1a73e8;
    font-size: 20px;
}

.technology-grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 12px;
    margin: 16px 0;
}

.tech-card {
    background: white;
    border: 1px solid #e0e0e0;
    border-radius: 6px;
    padding: 12px;
}

h2 {
    color: #1a73e8;
    font-size: 1.15em;
    margin: 0;
}

h3 {
    color: #1a73e8;
    font-size: 1em;
    margin: 8px 0;
}

.highlight {
    color: #1a73e8;
    font-weight: 600;
}
</style>
</head>
<body>

<div class="section-header">
    <h2>🔷 Part 2: RAG Model Implementation</h2>
</div>

<div class="explanation-box">
    <h3>Understanding the RAG Architecture</h3>
    <p>The Retrieval-Augmented Generation system enhances language model responses by incorporating document-specific context. Unlike standard LLMs that rely solely on training data, RAG systems actively search through your documents to find relevant information before generating responses.</p>
</div>

<div class="chart-container">
    <img src="https://www.dropbox.com/scl/fi/w7w1hfzgzdu46ydv9on00/RAG_Structure.png?rlkey=ef8r6nfdtbg3zvw90900w6rt8&dl=1" alt="RAG Architecture Overview" style="width: 70%; max-width: 70%;">
</div>

</body>
</html>


<div class="explanation-box" style="background-color: #f8f9fa;">
    <h3>How Retrieval Works</h3>
    <p>When a user submits a query, the system:</p>
    <ol style="margin: 8px 0 0 20px; padding: 0;">
        <li>Converts the query into an embedding vector using the same model</li>
        <li>Searches the FAISS index for the k most similar document chunks</li>
        <li>Passes these relevant chunks as context to the language model</li>
        <li>Generates a response grounded in the retrieved information</li>
    </ol>
    <p style="margin-top: 12px;">This approach ensures responses are based on your specific documents rather than general knowledge, significantly improving accuracy and relevance.</p>
</div>


<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.task-header {
    background-color: #e8f0fe;
    padding: 12px 14px;
    margin: 10px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

.task-details {
    background-color: #f8f9fa;
    padding: 12px;
    margin: 8px 0;
    border-radius: 4px;
    border: 1px solid #e0e0e0;
    font-size: 0.9em;
}

h3 {
    color: #1a73e8;
    font-size: 1.05em;
    margin: 0 0 8px 0;
}

.process-list {
    margin: 8px 0 0 0;
    padding-left: 20px;
}

.process-list li {
    margin: 4px 0;
    color: #444;
}

.highlight {
    color: #1a73e8;
    font-weight: 600;
}
</style>
</head>
<body>

<div class="task-header">
    <h3>📥 Task 1: Document Loading</h3>
    <div class="task-details">
        <p style="margin: 0 0 8px 0;">This task retrieves the course syllabus PDF from Dropbox and prepares it for RAG processing. The document undergoes three key transformations to enable efficient semantic search:</p>
        <ul class="process-list">
            <li><span class="highlight">Download:</span> Fetch the PDF file using the provided Dropbox URL</li>
            <li><span class="highlight">Load:</span> Extract text content from all pages using PyPDFLoader</li>
            <li><span class="highlight">Chunk:</span> Split the document into 1000-character segments with 200-character overlap to preserve context boundaries</li>
        </ul>
        <p style="margin: 8px 0 0 0;">The chunking strategy ensures that related information remains together while creating appropriately sized segments for embedding generation. The overlap prevents important context from being lost at chunk boundaries.</p>
    </div>
</div>

</body>
</html>

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#>> DOCUMENT LOADING AND PROCESSING
# This cell downloads the PDF from Dropbox, loads it into memory,
# and splits it into manageable chunks for vector search
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# 📦 Document Loaders
from langchain_community.document_loaders import PyPDFLoader  # Load content from PDFs
from langchain_community.document_loaders import TextLoader  # Load plain text files

# ✂️ Text Processing & Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter # Import the text splitter here


# Download PDF from Dropbox
dropbox_url = "https://www.dropbox.com/scl/fi/zedqrdppb6et1sm3s09r6/IE_5250_Applied_Generative_AI-2025.pdf?rlkey=tn3130kcd5o03twalmydn8t6p&e=1&dl=1"
pdf_path = "document.pdf"

response = requests.get(dropbox_url)
with open(pdf_path, "wb") as file:
    file.write(response.content)

# Load and process the PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Split into chunks for better retrieval accuracy
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
docs = text_splitter.split_documents(documents)

pretty_print(f"PDF successfully downloaded and processed\n{len(documents)} pages converted into {len(docs)} searchable chunks",
             title="📥 Document Loading Complete")

<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.task-header {
    background-color: #e8f0fe;
    padding: 10px 12px;
    margin: 10px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

h3 {
    color: #1a73e8;
    font-size: 1.05em;
    margin: 0;
}
</style>
</head>
<body>

<!-- Task 2 Header -->
<div class="task-header">
    <h3>🧮 Task 2: Embedding Generation & Vector Store Creation</h3>
</div>


</body>
</html>

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#>> EMBEDDING GENERATION AND VECTOR STORE CREATION
# This cell converts text chunks into vector embeddings using OpenAI's
# model and stores them in a FAISS index for fast similarity search
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# ✂️ Text Processing & Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Split text into chunks

# 📚 Vector Store & Embeddings
from langchain_community.vectorstores import FAISS  # FAISS for fast vector search

# Initialize embedding model
embedding_model = OpenAIEmbeddings(model=DEFAULT_EMBED_MODEL)

# Create FAISS vector store
vector_db = FAISS.from_documents(docs, embedding_model)

# Prepare sample chunks display
sample_chunks = []
for i in range(min(3, len(docs))):
    chunk_preview = docs[i].page_content[:150].strip()
    sample_chunks.append(f"• Chunk {i+1}: {chunk_preview}...")

sample_text = f"Embeddings successfully created for {len(docs)} chunks\n\nSample chunks:\n" + "\n".join(sample_chunks)
pretty_print(sample_text, title="🧠 Embedding Generation Complete")


<!DOCTYPE html>
<html>
<head>
<style>
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.4;
    color: #1a1a1a;
    max-width: 750px;
    margin: 0 auto;
    padding: 8px;
    font-size: 14px;
}

.task-header {
    background-color: #e8f0fe;
    padding: 10px 12px;
    margin: 10px 0;
    border-radius: 4px;
    border-left: 3px solid #1a73e8;
}

h3 {
    color: #1a73e8;
    font-size: 1.05em;
    margin: 0;
}
</style>
</head>
<body>


<!-- Task 3 Header -->
<div class="task-header">
    <h3>🔍 Task 3: Query and Retrieval</h3>
</div>

</body>
</html>

In [ ]:
!pip install langchain-classic